In [1]:
"""
A full tutorial is available here:
https://nichecompass.readthedocs.io/en/latest/tutorials/notebooks/mouse_cns_spatial_reference_mapping.html

Here we illustrate preaparation of our core dataset as a reference for mapping the validation data
"""

'\nA full tutorial is available here:\nhttps://nichecompass.readthedocs.io/en/latest/tutorials/notebooks/mouse_cns_spatial_reference_mapping.html\n\nHere we illustrate preaparation of our core dataset as a reference for mapping the validation data\n'

In [2]:
import os, sys
import random
import warnings
import logging
from datetime import datetime
# import gdown
import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import seaborn as sns
import squidpy as sq
#from matplotlib import gridspec
#from sklearn.preprocessing import MinMaxScaler
from re import sub
import numpy as np
import pickle

# from nichecompass.models import NicheCompass
# from nichecompass.utils import (add_gps_from_gp_dict_to_adata,
#                                 create_new_color_dict,
#                                 compute_communication_gp_network,
#                                 visualize_communication_gp_network,
#                                 extract_gp_dict_from_mebocost_ms_interactions,
#                                 #extract_gp_dict_from_mebocost_es_interactions,
#                                 extract_gp_dict_from_nichenet_lrt_interactions,
#                                 extract_gp_dict_from_omnipath_lr_interactions,
#                                 #filter_and_combine_gp_dict_gps,
#                                 filter_and_combine_gp_dict_gps_v2,
#                                 generate_enriched_gp_info_plots)


# %%


/software/cellgen/team298/ls34/vqgraph/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/software/cellgen/team298/ls34/vqgraph/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/software/cellgen/team298/ls34/vqgraph/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWarning)
/software/cellgen/team298/ls34/vqgraph/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  warnings.warn(msg, FutureWarning)
/software/cellgen/team298/ls34/vqgraph/lib/python3.10/site-packages/anndata/utils.py:429: 

# Set output dir

In [3]:
"""
make sure inside this path, you have the folders gene_annotations 
and gene_programs with the files
(available from https://github.com/Lotfollahi-lab/nichecompass/tree/main/data)
"""

handle='/lustre/scratch124/cellgen/haniffa/projects/developmental_fibroblasts/nobackup_output/nichecompasss/nichecompass/' 


# Set up reference and query (important part)

In [4]:
"""
load adata (includes reference and query)
- note that sample id is in adata.obs["Sample"]
- cell type is in adata.obs["Annotation"]

if using our adata as reference, then either:
1. remove all query samples (in query_batches below), or
2. add query samples to reference_batches, 

and then add your sample id's to query_batches
"""

ADATA_PATH= '/lustre/scratch126/cellgen/lotfollahi/ls34/nemo/adata_all.h5ad.clustered.clustered10.good.prenichecompass'
#'/nfs/team298/ls34/xenium_atlas/model_ALL_CLEAN_scanvi_ALL/adata_counts_integrated_final_colored.h5ad'

adata_vis=sc.read_h5ad(ADATA_PATH)  





In [5]:
adata_vis.shape

(1785932, 4952)

In [6]:
adata_vis.obs["sample"]=adata_vis.obs["info_id6"]
adata_vis.obs["sample"].value_counts()

sample
3D_BK25_week12-D2                                  43797
BK39_Week 12                                       32564
BK30_Day 14                                        32124
CE4-SKI-27-FO-1-S25-S29-S32                        30343
BK49_Past Lesional                                 29489
                                                   ...  
Baseline_resolved_CE6-SKI-20-FO-1-S22_replicate     6266
Baseline_resolved_CE4-SKI-27-FO-1-S22-B2            6206
BK21_Non-lesional Baseline                          6192
Baseline_resolved_CE4-SKI-27-FO-1-S22_replicate     6002
BK20_Lesional Baseline                              2348
Name: count, Length: 112, dtype: int64

In [7]:
adata_vis.obs["Annotation"].value_counts()

Annotation
KC3               239332
KC1               161801
F2: Universal     126533
Nonspecific       122537
Pericyte1          96550
                   ...  
TR1                    7
Satellite cell         4
Mac_CX3CR1+            4
ILC2                   2
Cartilage              1
Name: count, Length: 103, dtype: int64

In [8]:
"""
SPLIT INTO REFERENCE AND QUERY
"""
reference_batches =  ['Week 8 (resolved)_CE3-SKI-28-FO-4-S22-E1_a',
 'Lesional_CE5-SKI-28-FO-1-S22_replicate',
 'Week 8 (resolved)_CE3-SKI-28-FO-4-S22-E1_b',
 'BK39_Non-lesional Baseline',
 'Baseline_resolved_CE3-SKI-28-FO-1-S22-B1',
 'Week 8 (resolved)_CE5-SKI-27-FO-4-S22-E1',
 'BK23_Lesional Baseline',
 '3D_BK25_week12-D2',
 '3D_BK25_week12-D1orE1b',
 'Baseline_never_CE5-SKI-27-FO-2-S22_replicate',
 '3D_BK22_Lesional_baseline-C2',
 'BK51_Never Lesional',
 'BK20_Week 12',
 'BK30_Lesional Baseline',
 '3D_BK22_Lesional_baseline-A1',
 'Week 8 (resolved)_CE4-SKI-27-FO-3-S22-E2_b',
 'BK30_Week 12',
 '3D_BK22_Lesional_baseline-B1',
 'Lesional_Baseline_resolved_CE3-SKI-24-FO-1-S22_replicate',
 'Baseline_resolved_CE6-SKI-20-FO-1-S22-C2',
 'Baseline_never_CE5-SKI-27-FO-2-S22-C1',
 'BK49_wk8 Relapse',
 'Week 8 (resolved)_CE3-SKI-28-FO-4-S22_replicate',
 'BK21_Non-lesional Baseline',
 'BK18_Week 12',
 'BK24_Week 12',
 'Baseline_never_CE3-SKI-28-FO-2-S22_replicate',
 '3D_BK25_week12-B2',
 'BK39_Week 12',
 'BK22_Non-lesional Baseline',
 'BK27_Week 12',
 'Baseline_resolved_CE4-SKI-27-FO-1-S22-B2',
 'Baseline_resolved_CE6-SKI-20-FO-1-S22_replicate',
 'BK49_Past Lesional',
 'BK18_Non-lesional Baseline',
 '3D_BK22_Lesional_baseline-D1',
 'BK27_Lesional Baseline',
 'Baseline_resolved_CE6-SKI-28-FO-4-S22_replicate',
 'BK25_Lesional Baseline',
 'BK25_Week 12',
 'Lesional_CE6-SKI-28-FO-4-S22_replicate',
 'Baseline_resolved_CE4-SKI-27-FO-1-S22_replicate',
 'BK50_Never Lesional',
 'BK51_Past Lesional',
 'Baseline_resolved_CE5-SKI-27-FO-1-S22_replicate',
 'BK24_Non-lesional Baseline',
 'BK50_Past Lesional',
 'BK39_Lesional Baseline',
 'Lesional_CE5-SKI-28-FO-1-S22-A1',
 '3D_BK22_Lesional_baseline-B2',
 'BK25_Non-lesional Baseline',
 'BK20_Non-lesional Baseline',
 '3D_BK25_week12-D1orE1a',
 'Baseline_resolved_CE5-SKI-27-FO-1-S22-B1',
 '3D_BK25_week12-C1',
 'Week 8 (resolved)_CE6-SKI-28-FO-3-S22_replicate',
 'BK30_Day 14',
 '3D_BK22_Lesional_baseline-A2',
 'Week 8 (resolved)_CE6-SKI-28-FO-3-S22-E2',
 'BK18_Lesional Baseline',
 'Baseline_resolved_CE3-SKI-28-FO-1-S22_replicate',
 'Lesional_CE4-SKI-27-FO-4-S22-A2',
 'BK24_Lesional Baseline',
 'Baseline_never_CE4-SKI-21-FO-1-S22_replicate',
 'BK43_Never Lesional',
 'BK22_Lesional Baseline',
 '3D_BK25_week12-A2',
 'BK22_Week 12',
 'Week 8 (resolved)_CE4-SKI-27-FO-3-S22_replicate',
 'BK51_wk8 Relapse',
 'BK23_Non-lesional Baseline',
 'BK27_Non-lesional Baseline',
 'Week 8 (resolved)_CE4-SKI-27-FO-3-S22-E2_a',
 'Lesional_CE4-SKI-27-FO-4-S22_replicate',
 'Baseline_never_CE3-SKI-28-FO-2-S22-C1',
 'BK46_Never Lesional',
 'BK30_Non-lesional Baseline',
 'BK21_Week 12',
 'BK43_Past Lesional',
 'Week 8 (resolved)_CE5-SKI-27-FO-4-S22_replicate',
 'BK21_Lesional Baseline',
 'BK46_Past Lesional',
 '3D_BK25_week12-B1',
 'BK20_Lesional Baseline',
 'BK51_Past Lesional wk8 relaspe',
 'Baseline_never_CE4-SKI-21-FO-1-S22-C2',
 '3D_BK22_Lesional_baseline-D2',
 'Lesional_CE6-SKI-28-FO-4-S22-A1',
 'BK49_Past Lesional wk8 relaspe',
 'Baseline_resolved_CE6-SKI-28-FO-1-S22-B2',
 '3D_BK25_week12-C2',
 '3D_BK25_week12-A1',
 'Lesional_CE3-SKI-24-FO-1-S22-A1',
 'BK23_Week 12',
 'BK49_Never Lesional']
 
"""
replace these query batches with your samples
"""
query_batches = ['BK21-SKI-27-FO-1-S8-A3',
 'BK51-SKI-27-FO-2-S9-B2',
 'BK23-SKI-27-FO-1-S8-B1',
 'BK21-SKI-27-FO1-S11-C1',
 'BK27-SKI-27-FO-5-S9-D1',
 'CE3-SKI-28-FO-1-S25-E1',
 'BK21-SKI-27-FO-1-S13-C2',
 'BK22-SKI-27-FO-2-S7-A1',
 'BK30-SKI-28-FO-1-S6-B2',
 'BK39-SKI-27-FO-1-S8-D2',
 'CE3-SKI-28-FO-1-S25-D1',
 'BK30-SKI-28-FO-1-S14-C2',
 'BK27-SKI-27-FO-1-S6-C1',
 'CE4-SKI-27-FO-1-S25-S29-S32',
 'BK51-SKI-27-FO-2-S4-S8-S6',
 'BK23-SKI-27-FO-5-S9-A2',
 'CE3-SKI-28-FO-1-S28-D2']



In [9]:
adata_vis.obs["batch_nc"]=["reference" if x in reference_batches else "query" for x in adata_vis.obs["sample"]]

In [10]:
query_check = list(adata_vis[adata_vis.obs["batch_nc"]=="query"].obs["sample"].unique())
for x in query_batches:
    if x not in query_check:
        raise ValueError(f"Batch '{x}' not found in query samples.")

In [11]:
all_batches = reference_batches +query_batches


In [12]:
### from jimmy lee 
def select_slide2(adata, s, s_col='sample'):
    """ This function selects the data for one slide from the spatial anndata object.
    :param adata: Anndata object with multiple spatial experiments
    :param s: name of selected experiment
    :param s_col: column in adata.obs listing experiment name for each location
    """
    slide = adata[adata.obs[s_col].isin([s]), :]
#     s_keys = list(slide.uns['spatial'].keys())
#     s_spatial = np.array(s_keys)[[s in k for k in s_keys]][0]
#     slide.uns['spatial'] = {s_spatial: slide.uns['spatial'][s_spatial]}
    return slide

In [13]:
spatial_key = "spatial"
n_neighbors = 8
adj_key = "spatial_connectivities"

adata_batch_list = []
print("Processing reference batches...")
for batch in all_batches:
    print(f"Processing batch {batch}...")
    print("Loading data...")
    adata_batch = select_slide2(adata_vis, batch)
    print(f"Size {adata_batch.shape}")
    print("Computing spatial neighborhood graph...\n")
    # Compute (separate) spatial neighborhood graphs
    logging.info("sq.gr.spatial_neighbors")
    #try:
    sq.gr.spatial_neighbors(adata_batch,
                                coord_type="generic",
                                spatial_key=spatial_key,
                                n_neighs=n_neighbors)
    #except:
    #    continue
    print(f"Spatial neighbours done ## {adata_batch.shape}")

    # Make adjacency matrix symmetric
    adata_batch.obsp[adj_key] = (
        adata_batch.obsp[adj_key].maximum(
            adata_batch.obsp[adj_key].T))
    adata_batch_list.append(adata_batch)
print("List made: ...")
for x in adata_batch_list:
    print(x.shape)
adata_vis = ad.concat(adata_batch_list, join="inner")

Processing reference batches...
Processing batch Week 8 (resolved)_CE3-SKI-28-FO-4-S22-E1_a...
Loading data...
Size (13624, 4952)
Computing spatial neighborhood graph...



/software/cellgen/team298/ls34/vqgraph/lib/python3.10/site-packages/squidpy/gr/_utils.py:203: ImplicitModificationWarning: Setting element `.obsp['spatial_connectivities']` of view, initializing view as actual.
  obj[key] = data


Spatial neighbours done ## (13624, 4952)
Processing batch Lesional_CE5-SKI-28-FO-1-S22_replicate...
Loading data...
Size (12875, 4952)
Computing spatial neighborhood graph...

Spatial neighbours done ## (12875, 4952)
Processing batch Week 8 (resolved)_CE3-SKI-28-FO-4-S22-E1_b...
Loading data...


/software/cellgen/team298/ls34/vqgraph/lib/python3.10/site-packages/squidpy/gr/_utils.py:203: ImplicitModificationWarning: Setting element `.obsp['spatial_connectivities']` of view, initializing view as actual.
  obj[key] = data
/software/cellgen/team298/ls34/vqgraph/lib/python3.10/site-packages/squidpy/gr/_utils.py:203: ImplicitModificationWarning: Setting element `.obsp['spatial_connectivities']` of view, initializing view as actual.
  obj[key] = data


Size (10473, 4952)
Computing spatial neighborhood graph...

Spatial neighbours done ## (10473, 4952)
Processing batch BK39_Non-lesional Baseline...
Loading data...
Size (17911, 4952)
Computing spatial neighborhood graph...

Spatial neighbours done ## (17911, 4952)
Processing batch Baseline_resolved_CE3-SKI-28-FO-1-S22-B1...
Loading data...


...

Spatial neighbours done ## (28303, 4952)
Processing batch BK23-SKI-27-FO-5-S9-A2...
Loading data...
Size (24145, 4952)
Computing spatial neighborhood graph...



/software/cellgen/team298/ls34/vqgraph/lib/python3.10/site-packages/squidpy/gr/_utils.py:203: ImplicitModificationWarning: Setting element `.obsp['spatial_connectivities']` of view, initializing view as actual.
  obj[key] = data


Spatial neighbours done ## (24145, 4952)
Processing batch CE3-SKI-28-FO-1-S28-D2...
Loading data...
Size (12144, 4952)
Computing spatial neighborhood graph...



/software/cellgen/team298/ls34/vqgraph/lib/python3.10/site-packages/squidpy/gr/_utils.py:203: ImplicitModificationWarning: Setting element `.obsp['spatial_connectivities']` of view, initializing view as actual.
  obj[key] = data


Spatial neighbours done ## (12144, 4952)
List made: ...
(13624, 4952)
(12875, 4952)
(10473, 4952)
(17911, 4952)
(6479, 4952)
(19291, 4952)
(28588, 4952)
(43797, 4952)
(18084, 4952)
(11707, 4952)
(23723, 4952)
(16595, 4952)
(14996, 4952)
(14654, 4952)
(24058, 4952)
(13326, 4952)
(20283, 4952)
(21539, 4952)
(21899, 4952)
(6470, 4952)
(12056, 4952)
(12418, 4952)
(13293, 4952)
(6192, 4952)
(15133, 4952)
(12013, 4952)
(14824, 4952)
(18781, 4952)
(32564, 4952)
(9536, 4952)
(13475, 4952)
(6206, 4952)
(6266, 4952)
(29489, 4952)
(7668, 4952)
(25059, 4952)
(7959, 4952)
(10156, 4952)
(12295, 4952)
(15190, 4952)
(8658, 4952)
(6002, 4952)
(15320, 4952)
(14996, 4952)
(17309, 4952)
(12691, 4952)
(11110, 4952)
(17374, 4952)
(13116, 4952)
(26243, 4952)
(14348, 4952)
(6924, 4952)
(17867, 4952)
(17724, 4952)
(17270, 4952)
(8995, 4952)
(32124, 4952)
(20710, 4952)
(9120, 4952)
(17830, 4952)
(6504, 4952)
(21628, 4952)
(14199, 4952)
(9966, 4952)
(12988, 4952)
(22134, 4952)
(21014, 4952)
(10075, 4952)
(11607,

In [14]:
# Combine spatial neighborhood graphs as disconnected components
batch_connectivities = []
len_before_batch = 0
for i in range(len(adata_batch_list)):
    if i == 0: # first batch
        after_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[0].shape[0],
            (adata_vis.shape[0] -
            adata_batch_list[0].shape[0])))
        batch_connectivities.append(sp.hstack(
            (adata_batch_list[0].obsp[adj_key],
            after_batch_connectivities_extension)))
    elif i == (len(adata_batch_list) - 1): # last batch
        before_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[i].shape[0],
            (adata_vis.shape[0] -
            adata_batch_list[i].shape[0])))
        batch_connectivities.append(sp.hstack(
            (before_batch_connectivities_extension,
            adata_batch_list[i].obsp[adj_key])))
    else: # middle batches
        before_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[i].shape[0], len_before_batch))
        after_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[i].shape[0],
            (adata_vis.shape[0] -
            adata_batch_list[i].shape[0] -
            len_before_batch)))
        batch_connectivities.append(sp.hstack(
            (before_batch_connectivities_extension,
            adata_batch_list[i].obsp[adj_key],
            after_batch_connectivities_extension)))
    len_before_batch += adata_batch_list[i].shape[0]
adata_vis.obsp[adj_key] = sp.vstack(batch_connectivities)



In [15]:
sq.gr.spatial_autocorr(adata_vis, mode="moran", genes=adata_vis.var_names)


In [16]:
n_svg=1024
sv_genes = adata_vis.uns["moranI"].index[:n_svg].tolist()


In [17]:
adata_vis.var["spatially_variable"] = adata_vis.var_names.isin(sv_genes)
adata_vis.var["keep_gene"] = adata_vis.var["spatially_variable"]
adata_vis = adata_vis[:, adata_vis.var["keep_gene"] == True]
print(f"Keeping {len(adata_vis.var_names)} spatially variable, highly "
       "variable or gene program relevant genes.")


Keeping 1024 spatially variable, highly variable or gene program relevant genes.


In [18]:
import pickle
filepath = f"{handle}/svgenelist.pkl"
with open(filepath, "wb") as f:
    pickle.dump(sv_genes, f)

In [19]:
# sv_genes

In [20]:
# adata_vis

View of AnnData object with n_obs × n_vars = 1785932 × 1024
    obs: 'cell_id', 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'genomic_control_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'nucleus_count', 'segmentation_method', 'sample_id', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'pct_counts_in_top_10_genes', 'pct_counts_in_top_20_genes', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_150_genes', 'n_counts', 'xenium_id', 'xenium_id_recoded', 'Site_status', 'GSTT patient ID', 'Sanger patient ID', 'Drug', 'Responder', 'Timepoint', "Library type (CITE or 5'GEX/TCR)", 'Segmentation Y/N?', 'Sample ID', 'Xenium slide number', 'Xenium region number2', 'baseline_postrx', 'info_id', 'Annotation', 'NICHE_NAMES', 'batch', 'DonorID', 'leiden_res1', 'leiden_res2', 'new_annotation', 'new_annotation2', 'new_annotation3', 'annotation_new4', 'new_ann

In [21]:
adata_vis.write(ADATA_PATH + ".svg")  
print(ADATA_PATH + ".svg")

/software/cellgen/team298/ls34/vqgraph/lib/python3.10/site-packages/anndata/_core/anndata.py:1146: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  df[key] = c


/lustre/scratch126/cellgen/lotfollahi/ls34/nemo/adata_all.h5ad.clustered.clustered10.good.prenichecompass.svg


In [22]:
"""
this adata can now be used as input for the ref-query tutorial for
nichecompass
https://nichecompass.readthedocs.io/en/latest/tutorials/notebooks/mouse_cns_spatial_reference_mapping.html

we apply this in a python job in ../job_scripts/nichecompass_ref_mapping2.py

"""

'\nthis adata can now be used as input for the ref-query tutorial for\nnichecompass\nhttps://nichecompass.readthedocs.io/en/latest/tutorials/notebooks/mouse_cns_spatial_reference_mapping.html\n\nwe apply this in a python job in ../job_scripts/nichecompass_ref_mapping2.py\n\n'

In [23]:
stop

NameError: name 'stop' is not defined

# Check working

In [ ]:

ADATA_PATH= '/lustre/scratch126/cellgen/lotfollahi/ls34/nemo/adata_all.h5ad.clustered.clustered10.good.prenichecompass.svg'
adata_vis=sc.read_h5ad(ADATA_PATH)  
 

In [24]:

query_batches = list(adata_vis[adata_vis.obs["batch_nc"]=="query"].obs["sample"].unique())
reference_batches = list(adata_vis[adata_vis.obs["batch_nc"]=="reference"].obs["sample"].unique())


# %%

In [25]:
adata_vis.obs["batch_nc"].value_counts()

batch_nc
reference    1462847
query         323085
Name: count, dtype: int64

In [26]:
adata_batch_list = []
print("Processing reference batches...")
for batch in reference_batches:
    print(f"Processing batch {batch}...")
    print("Loading data...")
    adata_batch = select_slide2(adata_vis, batch)
#     print(f"Size {adata_batch.shape}")
#     print("Computing spatial neighborhood graph...\n")
#     # Compute (separate) spatial neighborhood graphs
#     logging.info("sq.gr.spatial_neighbors")
#     #try:
#     sq.gr.spatial_neighbors(adata_batch,
#                                 coord_type="generic",
#                                 spatial_key=spatial_key,
#                                 n_neighs=n_neighbors)
#     #except:
#     #    continue
#     print(f"Spatial neighbours done ## {adata_batch.shape}")

#     # Make adjacency matrix symmetric
#     adata_batch.obsp[adj_key] = (
#         adata_batch.obsp[adj_key].maximum(
#             adata_batch.obsp[adj_key].T))
    adata_batch_list.append(adata_batch)
print("List made: ...")
for x in adata_batch_list:
    print(x.shape)
adata_reference = ad.concat(adata_batch_list, join="inner")

Processing reference batches...
Processing batch Week 8 (resolved)_CE3-SKI-28-FO-4-S22-E1_a...
Loading data...
Processing batch Lesional_CE5-SKI-28-FO-1-S22_replicate...
Loading data...
Processing batch Week 8 (resolved)_CE3-SKI-28-FO-4-S22-E1_b...
Loading data...
Processing batch BK39_Non-lesional Baseline...
Loading data...
Processing batch Baseline_resolved_CE3-SKI-28-FO-1-S22-B1...
Loading data...
Processing batch Week 8 (resolved)_CE5-SKI-27-FO-4-S22-E1...
Loading data...
Processing batch BK23_Lesional Baseline...
Loading data...
Processing batch 3D_BK25_week12-D2...
Loading data...
Processing batch 3D_BK25_week12-D1orE1b...
Loading data...
Processing batch Baseline_never_CE5-SKI-27-FO-2-S22_replicate...
Loading data...
Processing batch 3D_BK22_Lesional_baseline-C2...
Loading data...
Processing batch BK51_Never Lesional...
Loading data...
Processing batch BK20_Week 12...
Loading data...
Processing batch BK30_Lesional Baseline...
Loading data...
Processing batch 3D_BK22_Lesional_ba

In [27]:
print("List made: ...", len(adata_batch_list))
for x in adata_batch_list:
    print(x.shape)

List made: ... 95
(13624, 1024)
(12875, 1024)
(10473, 1024)
(17911, 1024)
(6479, 1024)
(19291, 1024)
(28588, 1024)
(43797, 1024)
(18084, 1024)
(11707, 1024)
(23723, 1024)
(16595, 1024)
(14996, 1024)
(14654, 1024)
(24058, 1024)
(13326, 1024)
(20283, 1024)
(21539, 1024)
(21899, 1024)
(6470, 1024)
(12056, 1024)
(12418, 1024)
(13293, 1024)
(6192, 1024)
(15133, 1024)
(12013, 1024)
(14824, 1024)
(18781, 1024)
(32564, 1024)
(9536, 1024)
(13475, 1024)
(6206, 1024)
(6266, 1024)
(29489, 1024)
(7668, 1024)
(25059, 1024)
(7959, 1024)
(10156, 1024)
(12295, 1024)
(15190, 1024)
(8658, 1024)
(6002, 1024)
(15320, 1024)
(14996, 1024)
(17309, 1024)
(12691, 1024)
(11110, 1024)
(17374, 1024)
(13116, 1024)
(26243, 1024)
(14348, 1024)
(6924, 1024)
(17867, 1024)
(17724, 1024)
(17270, 1024)
(8995, 1024)
(32124, 1024)
(20710, 1024)
(9120, 1024)
(17830, 1024)
(6504, 1024)
(21628, 1024)
(14199, 1024)
(9966, 1024)
(12988, 1024)
(22134, 1024)
(21014, 1024)
(10075, 1024)
(11607, 1024)
(15245, 1024)
(19172, 1024)
(86

In [28]:
adata_reference = ad.concat(adata_batch_list, join="inner")

In [29]:
mapping_entity_key = "mapping_entity"
adata_reference.obs[mapping_entity_key] = "reference"


In [30]:
ADATA_PATH= '/lustre/scratch126/cellgen/lotfollahi/ls34/nemo/adata_all.h5ad.clustered.clustered10.good.prenichecompass.svg'


In [31]:
adata_reference.write(ADATA_PATH + ".reference")  
print(ADATA_PATH + ".reference")

/lustre/scratch126/cellgen/lotfollahi/ls34/nemo/adata_all.h5ad.clustered.clustered10.good.prenichecompass.svg.reference


In [32]:
adata_batch_list = []

for batch in query_batches:
    print(f"Processing batch {batch}...")
    print("Loading data...")
    adata_batch = select_slide2(adata_vis, batch)
    print("Computing spatial neighborhood graph...\n")
    # Compute (separate) spatial neighborhood graphs
#     sq.gr.spatial_neighbors(adata_batch,
#                             coord_type="generic",
#                             spatial_key=spatial_key,
#                             n_neighs=n_neighbors)
    
#     # Make adjacency matrix symmetric
#     adata_batch.obsp[adj_key] = (
#         adata_batch.obsp[adj_key].maximum(
#             adata_batch.obsp[adj_key].T))
    adata_batch_list.append(adata_batch)
adata_query = ad.concat(adata_batch_list, join="inner")

Processing batch BK21-SKI-27-FO-1-S8-A3...
Loading data...
Computing spatial neighborhood graph...

Processing batch BK51-SKI-27-FO-2-S9-B2...
Loading data...
Computing spatial neighborhood graph...

Processing batch BK23-SKI-27-FO-1-S8-B1...
Loading data...
Computing spatial neighborhood graph...

Processing batch BK21-SKI-27-FO1-S11-C1...
Loading data...
Computing spatial neighborhood graph...

Processing batch BK27-SKI-27-FO-5-S9-D1...
Loading data...
Computing spatial neighborhood graph...

Processing batch CE3-SKI-28-FO-1-S25-E1...
Loading data...
Computing spatial neighborhood graph...

Processing batch BK21-SKI-27-FO-1-S13-C2...
Loading data...
Computing spatial neighborhood graph...

Processing batch BK22-SKI-27-FO-2-S7-A1...
Loading data...
Computing spatial neighborhood graph...

Processing batch BK30-SKI-28-FO-1-S6-B2...
Loading data...
Computing spatial neighborhood graph...

Processing batch BK39-SKI-27-FO-1-S8-D2...
Loading data...
Computing spatial neighborhood graph...


In [ ]:

# for i in range(len(adata_batch_list)):
#     if i == 0: # first batch
#         after_batch_connectivities_extension = sp.csr_matrix(
#             (adata_batch_list[0].shape[0],
#             (adata_query.shape[0] -
#             adata_batch_list[0].shape[0])))
#         batch_connectivities.append(sp.hstack(
#             (adata_batch_list[0].obsp[adj_key],
#             after_batch_connectivities_extension)))
#     elif i == (len(adata_batch_list) - 1): # last batch
#         before_batch_connectivities_extension = sp.csr_matrix(
#             (adata_batch_list[i].shape[0],
#             (adata_query.shape[0] -
#             adata_batch_list[i].shape[0])))
#         batch_connectivities.append(sp.hstack(
#             (before_batch_connectivities_extension,
#             adata_batch_list[i].obsp[adj_key])))
#     else: # middle batches
#         before_batch_connectivities_extension = sp.csr_matrix(
#             (adata_batch_list[i].shape[0], len_before_batch))
#         after_batch_connectivities_extension = sp.csr_matrix(
#             (adata_batch_list[i].shape[0],
#             (adata_query.shape[0] -
#             adata_batch_list[i].shape[0] -
#             len_before_batch)))
#         batch_connectivities.append(sp.hstack(
#             (before_batch_connectivities_extension,
#             adata_batch_list[i].obsp[adj_key],
#             after_batch_connectivities_extension)))
#     len_before_batch += adata_batch_list[i].shape[0]
# adata_query.obsp[adj_key] = sp.vstack(batch_connectivities)
#batch_connectivities = []
#len_before_batch = 0


In [33]:
adata_query.obs[mapping_entity_key] = "query"

In [34]:
adata_query.shape

(323085, 1024)

In [35]:
adata_query.write(ADATA_PATH + ".query")  
print(ADATA_PATH + ".query")

/lustre/scratch126/cellgen/lotfollahi/ls34/nemo/adata_all.h5ad.clustered.clustered10.good.prenichecompass.svg.query


In [ ]:


### from jimmy lee 
def select_slide2(adata, s, s_col='sample'):
    """ This function selects the data for one slide from the spatial anndata object.
    :param adata: Anndata object with multiple spatial experiments
    :param s: name of selected experiment
    :param s_col: column in adata.obs listing experiment name for each location
    """
    slide = adata[adata.obs[s_col].isin([s]), :]
#     s_keys = list(slide.uns['spatial'].keys())
#     s_spatial = np.array(s_keys)[[s in k for k in s_keys]][0]
#     slide.uns['spatial'] = {s_spatial: slide.uns['spatial'][s_spatial]}
    return slide

adata_batch_list = []
print("Processing reference batches...")
for batch in reference_batches:
#     print(f"Processing batch {batch}...")
#     print("Loading data...")
#     adata_batch = select_slide2(adata_vis, batch)
#     print(f"Size {adata_batch.shape}")
#     print("Computing spatial neighborhood graph...\n")
#     # Compute (separate) spatial neighborhood graphs
#     logging.info("sq.gr.spatial_neighbors")
#     #try:
#     sq.gr.spatial_neighbors(adata_batch,
#                                 coord_type="generic",
#                                 spatial_key=spatial_key,
#                                 n_neighs=n_neighbors)
#     #except:
#     #    continue
#     print(f"Spatial neighbours done ## {adata_batch.shape}")

#     # Make adjacency matrix symmetric
#     adata_batch.obsp[adj_key] = (
#         adata_batch.obsp[adj_key].maximum(
#             adata_batch.obsp[adj_key].T))
    adata_batch_list.append(adata_batch)

print("List made: ...", len(adata_batch_list))

for x in adata_batch_list:
    print(x.shape)
adata_reference = ad.concat(adata_batch_list, join="inner")


# %%


# Combine spatial neighborhood graphs as disconnected components
# batch_connectivities = []
# len_before_batch = 0
# for i in range(len(adata_batch_list)):
#     if i == 0: # first batch
#         after_batch_connectivities_extension = sp.csr_matrix(
#             (adata_batch_list[0].shape[0],
#             (adata_reference.shape[0] -
#             adata_batch_list[0].shape[0])))
#         batch_connectivities.append(sp.hstack(
#             (adata_batch_list[0].obsp[adj_key],
#             after_batch_connectivities_extension)))
#     elif i == (len(adata_batch_list) - 1): # last batch
#         before_batch_connectivities_extension = sp.csr_matrix(
#             (adata_batch_list[i].shape[0],
#             (adata_reference.shape[0] -
#             adata_batch_list[i].shape[0])))
#         batch_connectivities.append(sp.hstack(
#             (before_batch_connectivities_extension,
#             adata_batch_list[i].obsp[adj_key])))
#     else: # middle batches
#         before_batch_connectivities_extension = sp.csr_matrix(
#             (adata_batch_list[i].shape[0], len_before_batch))
#         after_batch_connectivities_extension = sp.csr_matrix(
#             (adata_batch_list[i].shape[0],
#             (adata_reference.shape[0] -
#             adata_batch_list[i].shape[0] -
#             len_before_batch)))
#         batch_connectivities.append(sp.hstack(
#             (before_batch_connectivities_extension,
#             adata_batch_list[i].obsp[adj_key],
#             after_batch_connectivities_extension)))
#     len_before_batch += adata_batch_list[i].shape[0]
# adata_reference.obsp[adj_key] = sp.vstack(batch_connectivities)
adata_reference.obs[mapping_entity_key] = "reference"


